# LegalOps-Lite
### A working, heavily-commented mini-clone of an AI litigation-management system

This notebook teaches "the logic and code behind a website" like Provakil's
Enterprise Legal Management product by actually building a small, real
version of the same pattern: **scrape court data -> AI-extract structured
fields from messy legal text -> store/update case records -> compute
analytics -> render a dashboard**.

Every module below is REAL, working code (not pseudocode) with comments
explaining *why* each design choice was made, not just *what* the code
does -- that's the part that doesn't show up just from reading a live
website's marketing page.

**What this is NOT:** a copy of Provakil's actual source code (which is
private), or a production-ready legal product. It's a teaching artifact
that demonstrates the same architectural pattern at a small, understandable
scale.

**Run every cell top-to-bottom.** A dashboard HTML file is generated at the end.


## 0. Setup

In [1]:
import sys, subprocess
for pkg, import_name in [("beautifulsoup4", "bs4"), ("lxml", "lxml"), ("requests", "requests")]:
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet",
                         "--break-system-packages"], check=False)
print("Dependencies ready.")

Dependencies ready.


## 1. The data model (`models.py`)

**Lesson: before writing any scraping, AI, or dashboard code, decide what
your core objects ARE.** This is the single most important design decision
in a system like this -- every other module depends on this shape.

In [2]:
"""
Core data model.

Every real system like this starts here: before writing any scraping,
AI, or dashboard code, you decide what a "Case" actually IS in your
system -- what fields it has, what's required, what's optional. Get
this wrong and every downstream module (ingestion, AI extraction,
analytics, dashboard) has to work around it.

We use a dataclass rather than a dict for the same reason a real backend
would use an ORM model (e.g. a Django/SQLAlchemy model): it documents the
shape of the data in one place, gets you type hints and autocomplete, and
fails loudly (at construction time) if a required field is missing,
instead of failing silently later when some code tries to read a key
that doesn't exist.
"""
from dataclasses import dataclass, field
from datetime import date, datetime


@dataclass
class Case:
    """
    One matter being tracked. This mirrors what a real ELM system calls
    a "case" or "matter" -- the central object everything else attaches to
    (hearings, documents, spend entries, AI summaries).
    """
    case_no: str                 # e.g. "CS/4521/2025" -- the court's own case number
    title: str                   # e.g. "Sharma Textiles Pvt Ltd vs Rajendra Kumar"
    court: str                   # which court/bench this is listed before
    judge: str                   # presiding judge, if known
    filed_on: date                # when the case was filed -- used for "case aging"
    status: str = "Open"          # Open / Disposed / Stayed / Reserved for Judgment
    next_hearing: date | None = None
    last_purpose: str = ""        # free-text purpose from the latest cause-list entry
    next_action_on: str = "Unknown"  # "Petitioner" / "Respondent" / "Court" / "Unknown"
    tags: list[str] = field(default_factory=list)


@dataclass
class HearingEntry:
    """
    One row pulled from a daily cause-list -- a single hearing event for
    a single case on a single date. This is the raw, noisy unit that
    ingestion produces; a Case gets built/updated FROM a stream of these.
    """
    case_no: str
    case_title_raw: str   # title exactly as scraped, before cleanup
    court_raw: str         # court/judge string exactly as scraped
    purpose_raw: str       # purpose/order text exactly as scraped
    hearing_date: date


@dataclass
class SpendEntry:
    """
    One billing/expense line tied to a case -- mirrors Provakil's
    "stage-wise billing" feature (court fees, counsel fees, filing costs
    tied to a litigation stage like Filing / Hearing / Order).
    """
    case_no: str
    stage: str      # "Filing" / "Hearing" / "Order" / "Appeal"
    amount_inr: float
    incurred_on: date
    description: str = ""


## 2. The scraper (`causelist_scraper.py`) — REAL, working code

This is genuine BeautifulSoup parsing code. The only thing "simulated" is
the data source (an offline HTML fixture instead of a live court website)
-- the parsing logic itself is exactly what you'd write against a real
site. Notice how defensive it is: every lookup checks for missing/malformed
data, because real cause-list pages are messy and change without notice.

In [3]:
"""
Cause-list scraper.

THIS IS REAL, WORKING SCRAPING CODE -- it parses actual HTML with
BeautifulSoup. The only thing "simulated" is the source: instead of
pointing at a live court website (which would need that court's
permission, handle CAPTCHAs/rate-limits, and silently break the moment
they redesign their page), it parses a saved HTML fixture that mimics
the real messiness: inconsistent capitalization, extra whitespace,
merged "Court / Judge" columns, free-text "purpose" fields with no fixed
vocabulary.

This is also *why* a real system like Provakil needs 19,000+ separate
scrapers (or scraper configs) -- every court's website has a different
HTML structure, so this exact kind of defensive, per-site parsing logic
has to be written (and maintained!) once per court/forum. Swapping this
module's input from a local file to `requests.get(url).text` is the only
change needed to point it at a real page; the parsing logic underneath
stays the same.
"""
import re
from datetime import date

from bs4 import BeautifulSoup

# HearingEntry is defined above in this notebook


def _clean_text(raw: str | None) -> str:
    """
    Collapse whitespace and strip. Scraped HTML text is full of stray
    newlines, double spaces, and leading/trailing whitespace from how the
    page was indented -- never trust it to be clean.
    """
    if raw is None:
        return ""
    return re.sub(r"\s+", " ", raw).strip()


def _split_case_title(raw_title: str) -> str:
    """
    Normalize "X Vs. Y" / "X vs Y" / "X v. Y" / "X V Y" into one
    consistent "X vs Y" form. Real cause-lists use ALL of these
    interchangeably (sometimes on the same page), so this kind of
    regex-based normalization is a recurring theme in legal-tech scraping:
    the data is *semantically* consistent but *textually* chaotic.
    """
    cleaned = _clean_text(raw_title)
    # Matches "vs", "Vs.", "v.", "V", "versus" as a case-title separator,
    # case-insensitively, surrounded by whitespace.
    parts = re.split(r"\s+(?:vs\.?|v\.?|versus)\s+", cleaned, maxsplit=1, flags=re.IGNORECASE)
    if len(parts) == 2:
        return f"{parts[0].strip()} vs {parts[1].strip()}"
    return cleaned


def parse_causelist_html(html: str, hearing_date: date) -> list[HearingEntry]:
    """
    Parse one day's cause-list HTML into a list of HearingEntry records.

    This function is deliberately defensive: every cell lookup checks for
    None before reading .text, because a real cause-list page WILL
    occasionally have a malformed row (merged cells, a missing column, a
    stray ad banner injected into the table). A scraper that assumes
    perfect structure crashes in production the first time a court's
    webmaster tweaks their template -- which is often, and without notice.
    """
    soup = BeautifulSoup(html, "lxml")
    entries: list[HearingEntry] = []

    table = soup.find("table", class_="causelist-table")
    if table is None:
        # Defensive: if the page structure changed and we can't find our
        # expected table, fail loudly with an empty result rather than
        # silently returning garbage or raising deep inside pandas/etc.
        return entries

    rows = table.find_all("tr", class_="case-row")
    for row in rows:
        cells = row.find_all("td")
        if len(cells) < 6:
            # A malformed row (merged cell, missing column) -- skip rather
            # than crash the whole batch over one bad row.
            continue

        case_no = _clean_text(cells[1].text)
        title_raw = _split_case_title(cells[2].text)
        court_raw = _clean_text(cells[3].text)
        purpose_raw = _clean_text(cells[5].text)

        if not case_no:
            continue  # a row with no case number is useless to us

        entries.append(HearingEntry(
            case_no=case_no,
            case_title_raw=title_raw,
            court_raw=court_raw,
            purpose_raw=purpose_raw,
            hearing_date=hearing_date,
        ))

    return entries


def extract_judge_name(court_raw: str) -> tuple[str, str]:
    """
    Split a merged "Court No. 7 - Hon'ble Smt. Justice A. Verma" string
    into (court, judge). Real cause-lists almost never give you these as
    separate fields -- they're merged into one free-text column, so
    splitting them back apart is its own small parsing problem.
    """
    if " - " in court_raw:
        court_part, judge_part = court_raw.split(" - ", 1)
        return court_part.strip(), judge_part.strip()
    return court_raw, ""


## 3. The AI extraction module (`ai_extract.py`) — the "AI" in this AI product

This is what Provakil's "Summarize Orders" and "Review Case Progress"
features are doing under the hood: turning free-text legal language into
structured fields. **Dual-mode design, same reasoning as before**: fast
free rules for the common cases, an optional LLM call only for the cases
rules can't confidently classify -- which is also how a real system at
scale controls LLM cost (you do NOT want to pay for an LLM call on every
single one of 80,000 daily cause-list rows when 80% follow predictable
boilerplate language).

In [4]:
"""
AI-based court-order field extraction and next-action classification.

This is the module that does what Provakil's "Summarize Orders & Petitions"
and "Review Case Progress & Next Actions" features do: take messy,
free-text legal text and turn it into structured fields (Court, Parties,
Order Type, who the next action is on).

Same dual-mode design as before, for the same reason: a rule-based path
that always works offline, and an optional LLM path for the cases rules
can't confidently handle. In production, a real system almost always
needs BOTH -- rules for the 80% of cases that follow predictable
boilerplate language, and an LLM for the 20% that use unusual phrasing a
regex was never written to catch. Relying on the LLM alone for every
single document is also slower and costlier at the scale Provakil
operates at (80,000-90,000 cause-list entries a day) -- which is exactly
why a real system reaches for rules first and reserves the LLM call for
cases the rules flag as uncertain.
"""
import os
import re
from dataclasses import dataclass


@dataclass
class ExtractedOrderFields:
    court: str
    judge: str
    parties: str
    order_type: str
    next_action_on: str   # "Petitioner" / "Respondent" / "Court" / "Unknown"
    confidence: str        # "rule_based_high" / "rule_based_low" / "llm"


# ---------------------------------------------------------------------------
# Mode 1: rule-based classification (fast, free, always available)
# ---------------------------------------------------------------------------

# Each pattern is checked against the lower-cased purpose text. Order
# matters: more specific patterns are checked first so a purpose string
# that matches multiple patterns gets the most informative label.
_ORDER_TYPE_PATTERNS: list[tuple[str, str]] = [
    (r"reserved for judg?e?ment", "Judgment Reserved"),
    (r"final argument", "Final Arguments"),
    (r"counsel absent|next date to be fixed", "Adjourned (Counsel Absent)"),
    (r"for orders?\b", "For Orders"),
    (r"for hearing", "For Hearing"),
    (r"rejoinder", "Rejoinder Stage"),
    (r"reply", "Reply Stage"),
]

# Who the ball is in court with, inferred from the purpose text. This is
# a genuinely hard NLP problem in general (it requires understanding who
# "petitioner" and "respondent" even are in THIS case, and what action
# verb applies to whom) -- the rule-based version below only catches the
# common, explicit phrasings; anything subtler should fall through to the
# LLM path.
_NEXT_ACTION_PATTERNS: list[tuple[str, str]] = [
    (r"petitioner.{0,25}(file|submit|appear|argue)", "Petitioner"),
    (r"respondent.{0,25}(file|submit|appear|argue)", "Respondent"),
    (r"applicant.{0,25}(file|submit|appear|argue)", "Petitioner"),
    (r"reserved for judg?e?ment", "Court"),
]


def rule_based_extract(court_raw: str, judge_raw: str, title_raw: str, purpose_raw: str) -> ExtractedOrderFields:
    purpose_lower = purpose_raw.lower()

    order_type = "Unclassified"
    for pattern, label in _ORDER_TYPE_PATTERNS:
        if re.search(pattern, purpose_lower):
            order_type = label
            break

    next_action = "Unknown"
    for pattern, label in _NEXT_ACTION_PATTERNS:
        if re.search(pattern, purpose_lower):
            next_action = label
            break

    # Confidence is "low" whenever we couldn't classify at all -- this is
    # exactly the signal a real pipeline uses to decide "send this one to
    # the LLM instead of trusting the rule-based guess."
    confidence = "rule_based_high" if (order_type != "Unclassified" and next_action != "Unknown") else "rule_based_low"

    return ExtractedOrderFields(
        court=court_raw,
        judge=judge_raw,
        parties=title_raw,
        order_type=order_type,
        next_action_on=next_action,
        confidence=confidence,
    )


# ---------------------------------------------------------------------------
# Mode 2: optional LLM-backed extraction (for cases rules can't confidently handle)
# ---------------------------------------------------------------------------

_EXTRACTION_SYSTEM_PROMPT = (
    "You are a legal-operations assistant. Given a single cause-list row from an "
    "Indian court (case title, court/judge, and a free-text 'purpose' field), extract: "
    "order_type (a short label like 'For Hearing', 'Judgment Reserved', 'Reply Stage'), "
    "and next_action_on (exactly one of: Petitioner, Respondent, Court, Unknown -- whoever "
    "the purpose text indicates must act next). "
    "Respond ONLY with a JSON object with keys 'order_type' and 'next_action_on', nothing else."
)


def llm_extract(court_raw: str, judge_raw: str, title_raw: str, purpose_raw: str,
                 api_key: str | None = None, model: str = "claude-sonnet-4-6",
                 timeout: int = 30) -> ExtractedOrderFields:
    """
    Calls the Anthropic API to classify a purpose string the rule-based
    path couldn't confidently handle. Raises on failure -- callers should
    catch and fall back to the rule-based result.
    """
    import json
    import requests

    key = api_key or os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise RuntimeError("No Anthropic API key found (pass api_key= or set ANTHROPIC_API_KEY).")

    user_content = (
        f"Case title: {title_raw}\nCourt/Judge: {court_raw}\nPurpose text: {purpose_raw}"
    )
    payload = {
        "model": model,
        "max_tokens": 150,
        "system": _EXTRACTION_SYSTEM_PROMPT,
        "messages": [{"role": "user", "content": user_content}],
    }
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": key, "anthropic-version": "2023-06-01", "content-type": "application/json"},
        json=payload, timeout=timeout,
    )
    resp.raise_for_status()
    content = resp.json()["content"]
    text = "".join(b.get("text", "") for b in content if b.get("type") == "text").strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text.strip())  # strip markdown fences if present
    parsed = json.loads(text)

    return ExtractedOrderFields(
        court=court_raw, judge=judge_raw, parties=title_raw,
        order_type=parsed.get("order_type", "Unclassified"),
        next_action_on=parsed.get("next_action_on", "Unknown"),
        confidence="llm",
    )


def extract(court_raw: str, judge_raw: str, title_raw: str, purpose_raw: str,
            api_key: str | None = None, use_llm_on_low_confidence: bool = True) -> ExtractedOrderFields:
    """
    Main entry point: tries the fast rule-based path first. Only falls
    through to the (slower, costlier) LLM path if the rules produced a
    low-confidence result AND an API key is available -- otherwise returns
    the rule-based guess as-is. This mirrors how a real high-volume
    pipeline controls LLM spend: never call the model when a cheap
    heuristic already got a confident answer.
    """
    result = rule_based_extract(court_raw, judge_raw, title_raw, purpose_raw)

    key = api_key or os.environ.get("ANTHROPIC_API_KEY")
    if result.confidence == "rule_based_low" and use_llm_on_low_confidence and key:
        try:
            return llm_extract(court_raw, judge_raw, title_raw, purpose_raw, api_key=key)
        except Exception:
            pass  # silently keep the rule-based guess if the LLM call fails

    return result


## 4. The case store / orchestrator (`case_store.py`)

**Lesson: the "service layer" pattern.** The scraper doesn't know the AI
module exists; the AI module doesn't know the dashboard exists. This
module is the only place that wires them together. This separation is
why real engineering teams can have different people/teams own scraping,
AI, and frontend without constantly breaking each other's code.

In [5]:
"""
Case store / orchestrator.

This is the module that actually wires the others together -- the same
role a backend "service layer" plays in a real application: the scraper
and the AI-extraction module don't know about each other, and neither
should know about the HTML dashboard. This module sits in the middle and
coordinates: take raw scraped entries -> run AI extraction on each ->
create or update the matching Case record.

This separation (ingestion / extraction / storage are three separate,
independently testable modules) is itself the main "software engineering"
lesson in a system like this -- not any single clever algorithm.
"""
from datetime import date

# extract_judge_name, Case, HearingEntry, and extract() are all defined above in this notebook


class CaseStore:
    """
    An in-memory case store. A real system would back this with a
    database (Postgres, most likely, given the relational structure of
    cases/hearings/spend) -- but the *logic* of "given a new hearing
    entry, create or update the matching case" is identical either way.
    Swapping this for a real DB-backed store later would not change any
    of the methods below, only how __init__ and _save() are implemented.
    """

    def __init__(self):
        self._cases: dict[str, Case] = {}

    def ingest_hearing_entries(self, entries: list[HearingEntry], api_key: str | None = None) -> list[Case]:
        """
        The main pipeline: for each raw scraped row, run AI extraction,
        then create a new Case (if we've never seen this case_no before)
        or update the existing one with the latest hearing info.
        """
        updated: list[Case] = []
        for entry in entries:
            court, judge = extract_judge_name(entry.court_raw)
            fields = extract(
                court_raw=court, judge_raw=judge,
                title_raw=entry.case_title_raw, purpose_raw=entry.purpose_raw,
                api_key=api_key,
            )

            existing = self._cases.get(entry.case_no)
            if existing is None:
                case = Case(
                    case_no=entry.case_no,
                    title=entry.case_title_raw,
                    court=court,
                    judge=judge,
                    filed_on=entry.hearing_date,  # best guess until we see an earlier entry
                    status="Open",
                    next_hearing=entry.hearing_date,
                    last_purpose=entry.purpose_raw,
                    next_action_on=fields.next_action_on,
                )
            else:
                case = existing
                case.next_hearing = entry.hearing_date
                case.last_purpose = entry.purpose_raw
                case.next_action_on = fields.next_action_on
                if entry.hearing_date < case.filed_on:
                    case.filed_on = entry.hearing_date  # earlier record found -> refine filing date

            if fields.order_type == "Judgment Reserved":
                case.status = "Reserved for Judgment"

            self._cases[entry.case_no] = case
            updated.append(case)

        return updated

    def all_cases(self) -> list[Case]:
        return list(self._cases.values())

    def get(self, case_no: str) -> Case | None:
        return self._cases.get(case_no)


## 5. Analytics (`analytics.py`) — the "data analyst" layer

Worth being explicit about: **this module uses zero machine learning.**
Case aging, status counts, spend totals -- it's all date-math and
group-by-aggregate, the same thing a `pandas.groupby()` or a SQL `GROUP BY`
does. A lot of what gets marketed as "AI-powered analytics" on a landing
page is actually plain aggregation sitting next to a genuinely-AI feature
(the extractor above) in the same product. Knowing which parts of a
system are real ML versus well-organized math is itself a useful skill.

In [6]:
"""
Analytics module.

This is the "data analyst" layer of the system -- notably, it uses NO
machine learning at all. "Case aging," "status distribution," and "spend
by stage" are just date-math and group-by-aggregate, the same operations
a real BI tool (or a pandas .groupby()) would do. It's worth being
explicit about this: a lot of what gets marketed as "AI-powered
analytics" is actually plain aggregation sitting next to a genuinely
AI-powered feature (the order summarizer) in the same product -- knowing
which parts of a system are "real ML" versus "just well-organized math"
is itself a useful skill to be able to point to.
"""
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import date

# Case, SpendEntry are defined above in this notebook


def case_aging_days(case: Case, as_of: date | None = None) -> int:
    """Days since filing -- the simplest possible 'case aging' metric."""
    reference = as_of or date.today()
    return (reference - case.filed_on).days


def aging_buckets(cases: list[Case], as_of: date | None = None) -> dict[str, int]:
    """
    Groups cases into aging buckets (0-90 / 91-365 / 365+ days), exactly
    the kind of bucketed histogram a real "case aging dashboard" shows.
    This is pure arithmetic + a lookup table -- no model involved.
    """
    buckets = {"0-90 days": 0, "91-365 days": 0, "365+ days": 0}
    for case in cases:
        age = case_aging_days(case, as_of)
        if age <= 90:
            buckets["0-90 days"] += 1
        elif age <= 365:
            buckets["91-365 days"] += 1
        else:
            buckets["365+ days"] += 1
    return buckets


def status_distribution(cases: list[Case]) -> dict[str, int]:
    """Simple frequency count -- a Counter, nothing more."""
    return dict(Counter(c.status for c in cases))


def next_action_distribution(cases: list[Case]) -> dict[str, int]:
    return dict(Counter(c.next_action_on for c in cases))


def spend_by_stage(entries: list[SpendEntry]) -> dict[str, float]:
    """Total INR spent per litigation stage -- a group-by-sum."""
    totals: dict[str, float] = defaultdict(float)
    for e in entries:
        totals[e.stage] += e.amount_inr
    return dict(totals)


def spend_by_case(entries: list[SpendEntry]) -> dict[str, float]:
    totals: dict[str, float] = defaultdict(float)
    for e in entries:
        totals[e.case_no] += e.amount_inr
    return dict(totals)


@dataclass
class PortfolioSummary:
    total_cases: int
    status_counts: dict[str, int]
    aging_buckets: dict[str, int]
    next_action_counts: dict[str, int]
    total_spend_inr: float
    spend_by_stage: dict[str, float]


def summarize_portfolio(cases: list[Case], spend_entries: list[SpendEntry],
                         as_of: date | None = None) -> PortfolioSummary:
    """
    Rolls everything above into one summary object -- this is exactly
    what feeds a dashboard's top-row summary cards.
    """
    stage_totals = spend_by_stage(spend_entries)
    return PortfolioSummary(
        total_cases=len(cases),
        status_counts=status_distribution(cases),
        aging_buckets=aging_buckets(cases, as_of),
        next_action_counts=next_action_distribution(cases),
        total_spend_inr=sum(stage_totals.values()),
        spend_by_stage=stage_totals,
    )


## 6. The dashboard (frontend) — `dashboard.py`

Notice this module never computes anything itself -- it only formats data it's handed. That's what lets a real frontend and backend evolve independently.

In [7]:
"""
HTML dashboard generator.

This is the "frontend" layer -- notice it takes already-computed Python
objects (Case lists, PortfolioSummary) and only formats them. It does NOT
recompute anything, query a database, or call the AI module itself. That
separation -- dashboard code never does business logic, it only renders
what it's handed -- is exactly why a real frontend (React, in Provakil's
actual case) and a real backend can be built and changed independently of
each other. A chart library changing doesn't touch your case-aging math;
changing how case-aging is computed doesn't touch any HTML.
"""
import html


def _esc(s) -> str:
    return html.escape(str(s), quote=True)


def _bar_row(label: str, value: int, max_value: int, color: str = "#2563eb") -> str:
    """Renders one horizontal bar -- used for status/aging/next-action breakdowns."""
    pct = 0 if max_value == 0 else round(100 * value / max_value)
    return f"""
    <div class="bar-row">
      <div class="bar-label">{_esc(label)}</div>
      <div class="bar-track"><div class="bar-fill" style="width:{pct}%;background:{color}"></div></div>
      <div class="bar-value">{value}</div>
    </div>"""


def render_dashboard_html(cases: list, summary, title: str = "LegalOps-Lite Dashboard") -> str:
    status_colors = {
        "Open": "#2563eb", "Reserved for Judgment": "#7c3aed",
        "Disposed": "#16a34a", "Stayed": "#d97706",
    }
    max_status = max(summary.status_counts.values(), default=1)
    status_rows = "".join(
        _bar_row(k, v, max_status, status_colors.get(k, "#64748b"))
        for k, v in summary.status_counts.items()
    )

    max_age = max(summary.aging_buckets.values(), default=1)
    aging_rows = "".join(_bar_row(k, v, max_age, "#0891b2") for k, v in summary.aging_buckets.items())

    max_action = max(summary.next_action_counts.values(), default=1)
    action_colors = {"Petitioner": "#ea580c", "Respondent": "#2563eb", "Court": "#7c3aed", "Unknown": "#64748b"}
    action_rows = "".join(
        _bar_row(k, v, max_action, action_colors.get(k, "#64748b"))
        for k, v in summary.next_action_counts.items()
    )

    max_spend = max(summary.spend_by_stage.values(), default=1)
    spend_rows = "".join(
        _bar_row(k, int(v), int(max_spend), "#16a34a")
        for k, v in summary.spend_by_stage.items()
    )

    case_rows = "".join(
        f"""<tr>
          <td class="mono">{_esc(c.case_no)}</td>
          <td>{_esc(c.title)}</td>
          <td>{_esc(c.court)}</td>
          <td><span class="badge" style="background:{status_colors.get(c.status, '#64748b')}">{_esc(c.status)}</span></td>
          <td><span class="badge" style="background:{action_colors.get(c.next_action_on, '#64748b')}">{_esc(c.next_action_on)}</span></td>
          <td class="small">{_esc(c.last_purpose)}</td>
        </tr>""" for c in cases
    )

    return f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{_esc(title)}</title>
<style>{_BASE_CSS}</style></head>
<body><div class="wrap">
  <header>
    <div><h1>{_esc(title)}</h1>
    <div class="sub">A working mini-clone of an AI litigation-management dashboard &mdash; built to teach the pattern, not a product.</div></div>
    <div class="card big-stat"><div class="k">Cases Tracked</div><div class="v">{summary.total_cases}</div></div>
  </header>

  <section class="grid-3">
    <div class="panel"><h2>Case Status</h2>{status_rows}</div>
    <div class="panel"><h2>Case Aging</h2>{aging_rows}</div>
    <div class="panel"><h2>Next Action On</h2>{action_rows}</div>
  </section>

  <section class="panel">
    <h2>Legal Spend by Stage &mdash; Total &#8377;{summary.total_spend_inr:,.0f}</h2>
    {spend_rows}
  </section>

  <section class="panel">
    <h2>Digital Case Workspace</h2>
    <table><thead><tr><th>Case No.</th><th>Title</th><th>Court</th><th>Status</th><th>Next Action On</th><th>Latest Purpose (AI-extracted)</th></tr></thead>
    <tbody>{case_rows}</tbody></table>
  </section>

  <footer>Generated by LegalOps-Lite &mdash; a teaching project, not a production legal product.</footer>
</div></body></html>"""


_BASE_CSS = """
:root{--bg:#f8fafc;--panel:#fff;--border:#e2e8f0;--text:#0f172a;--muted:#64748b;}
@media (prefers-color-scheme: dark){
  :root{--bg:#0b1220;--panel:#111a2b;--border:#1e293b;--text:#e2e8f0;--muted:#94a3b8;}
}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--text);font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;}
.wrap{max-width:1100px;margin:0 auto;padding:24px;}
header{display:flex;justify-content:space-between;align-items:center;margin-bottom:20px;flex-wrap:wrap;gap:12px;}
h1{margin:0;font-size:1.5rem;} .sub{color:var(--muted);font-size:.85rem;margin-top:4px;max-width:500px;}
.big-stat{text-align:center;min-width:140px;} .big-stat .v{font-size:2rem;font-weight:700;color:#2563eb;}
.grid-3{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:20px;}
@media (max-width:800px){.grid-3{grid-template-columns:1fr;}}
.card{background:var(--panel);border:1px solid var(--border);border-radius:10px;padding:14px;}
.k{color:var(--muted);font-size:.75rem;text-transform:uppercase;letter-spacing:.04em;}
.panel{background:var(--panel);border:1px solid var(--border);border-radius:10px;padding:18px;margin-bottom:20px;}
.panel h2{margin:0 0 14px;font-size:1.02rem;}
.bar-row{display:grid;grid-template-columns:130px 1fr 36px;align-items:center;gap:10px;margin-bottom:10px;font-size:.85rem;}
.bar-track{background:var(--border);border-radius:6px;height:10px;overflow:hidden;}
.bar-fill{height:100%;border-radius:6px;}
table{width:100%;border-collapse:collapse;font-size:.85rem;}
th,td{text-align:left;padding:8px 10px;border-bottom:1px solid var(--border);vertical-align:top;}
th{color:var(--muted);font-weight:600;font-size:.72rem;text-transform:uppercase;}
.badge{color:#fff;padding:2px 8px;border-radius:6px;font-size:.7rem;font-weight:600;white-space:nowrap;}
.mono{font-family:ui-monospace,Consolas,monospace;} .small{font-size:.78rem;color:var(--muted);}
footer{text-align:center;color:var(--muted);font-size:.8rem;padding:20px 0;}
"""


## 7. Load the sample cause-list fixture

A realistic mock of a messy, inconsistent Indian court cause-list page
(embedded as base64 so this notebook needs no external files).

In [8]:
import base64
_FIXTURE_B64 = "PCFET0NUWVBFIGh0bWw+CjxodG1sPgo8aGVhZD48dGl0bGU+RGlzdHJpY3QgQ291cnQgQ2F1c2UgTGlzdCAtIDA4LU5vdi0yMDI2PC90aXRsZT48L2hlYWQ+Cjxib2R5Pgo8IS0tCiAgVGhpcyBmaXh0dXJlIGRlbGliZXJhdGVseSBtaW1pY3MgdGhlIG1lc3NpbmVzcyBvZiBhIHJlYWwgSW5kaWFuIGNvdXJ0CiAgd2Vic2l0ZSdzIGNhdXNlLWxpc3QgcGFnZTogaW5jb25zaXN0ZW50IGNhcGl0YWxpemF0aW9uLCBleHRyYSB3aGl0ZXNwYWNlLAogIG1lcmdlZCB0YWJsZSBjZWxscywgYW5kIG5vIGNsZWFuIEpTT04gQVBJLiBBIHJlYWwgc2NyYXBlciBoYXMgdG8gYmUKICBkZWZlbnNpdmUgYWdhaW5zdCBleGFjdGx5IHRoaXMga2luZCBvZiBtZXNzLCB3aGljaCBpcyB3aHkgdGhlIHBhcnNlcgogIGJlbG93IG5ldmVyIGFzc3VtZXMgYSBmaWVsZCBpcyBwcmVzZW50IG9yIGNsZWFuLgotLT4KPHRhYmxlIGNsYXNzPSJjYXVzZWxpc3QtdGFibGUiIGJvcmRlcj0iMSI+CiAgPHRyIGNsYXNzPSJoZWFkZXItcm93Ij4KICAgIDx0aD5Tci5ObzwvdGg+PHRoPkNhc2UgTm8uPC90aD48dGg+Q2FzZSBUaXRsZTwvdGg+PHRoPkNvdXJ0IC8gSnVkZ2U8L3RoPjx0aD5JdGVtPC90aD48dGg+UHVycG9zZTwvdGg+CiAgPC90cj4KICA8dHIgY2xhc3M9ImNhc2Utcm93Ij4KICAgIDx0ZD4xPC90ZD4KICAgIDx0ZD4gIENTLzQ1MjEvMjAyNSAgPC90ZD4KICAgIDx0ZD5NL3MgU2hhcm1hIFRleHRpbGVzIFB2dCBMdGQgIFZzLiAgUmFqZW5kcmEgS3VtYXI8L3RkPgogICAgPHRkPkNvdXJ0IE5vLiA3IC0gSG9uJ2JsZSBTbXQuIEp1c3RpY2UgQS4gVmVybWE8L3RkPgogICAgPHRkPjEyPC90ZD4KICAgIDx0ZD5Gb3IgSGVhcmluZyAtIFBldGl0aW9uZXIgdG8gZmlsZSByZWpvaW5kZXI8L3RkPgogIDwvdHI+CiAgPHRyIGNsYXNzPSJjYXNlLXJvdyI+CiAgICA8dGQ+MjwvdGQ+CiAgICA8dGQ+Q1MvMzM5MC8yMDI0PC90ZD4KICAgIDx0ZD5HbG9iYWwgTG9naXN0aWNzIEx0ZCB2cyBNZXRybyBXYXJlaG91c2luZyBDby48L3RkPgogICAgPHRkPkNvdXJ0IE5vLiAzIC0gSG9uJ2JsZSBTaHJpIEp1c3RpY2UgSy4gTWVodGE8L3RkPgogICAgPHRkPjU8L3RkPgogICAgPHRkPkZvciBPcmRlcnMgLSBSZXNwb25kZW50IHRvIGZpbGUgcmVwbHkgd2l0aGluIHR3byB3ZWVrczwvdGQ+CiAgPC90cj4KICA8dHIgY2xhc3M9ImNhc2Utcm93Ij4KICAgIDx0ZD4zPC90ZD4KICAgIDx0ZD4gQVJCLzExMi8yMDI2PC90ZD4KICAgIDx0ZD5Ob3J0aHN0YXIgRmludGVjaCBQdnQuIEx0ZC4gVnMgQW5pbCBEZXNobXVraCAmYW1wOyBBbnIuPC90ZD4KICAgIDx0ZD5Db3VydCBOby4gNyAtIEhvbidibGUgU210LiBKdXN0aWNlIEEuIFZlcm1hPC90ZD4KICAgIDx0ZD4xOTwvdGQ+CiAgICA8dGQ+RmluYWwgQXJndW1lbnRzIC0gUmVzZXJ2ZWQgZm9yIEp1ZGdtZW50PC90ZD4KICA8L3RyPgogIDx0ciBjbGFzcz0iY2FzZS1yb3ciPgogICAgPHRkPjQ8L3RkPgogICAgPHRkPkNTLzk5ODEvMjAyMzwvdGQ+CiAgICA8dGQ+ICBCcmlnaHQgRnV0dXJlIE1hbnVmYWN0dXJpbmcgICB2LiAgU3RhdGUgRWxlY3RyaWNpdHkgQm9hcmQgPC90ZD4KICAgIDx0ZD5Db3VydCBOby4gMSAtIEhvbidibGUgU2hyaSBKdXN0aWNlIFIuIE5haXI8L3RkPgogICAgPHRkPjI8L3RkPgogICAgPHRkPkZvciBIZWFyaW5nIC0gUGV0aXRpb25lcidzIGNvdW5zZWwgYWJzZW50LCBuZXh0IGRhdGUgdG8gYmUgZml4ZWQ8L3RkPgogIDwvdHI+CjwvdGFibGU+CjwvYm9keT4KPC9odG1sPgo="
causelist_html = base64.b64decode(_FIXTURE_B64).decode("utf-8")
print(causelist_html[:500], "...")

<!DOCTYPE html>
<html>
<head><title>District Court Cause List - 08-Nov-2026</title></head>
<body>
<!--
  This fixture deliberately mimics the messiness of a real Indian court
  website's cause-list page: inconsistent capitalization, extra whitespace,
  merged table cells, and no clean JSON API. A real scraper has to be
  defensive against exactly this kind of mess, which is why the parser
  below never assumes a field is present or clean.
-->
<table class="causelist-table" border="1">
  <tr clas ...


## 8. Run the scraper on the fixture

In [9]:
from datetime import date

entries = parse_causelist_html(causelist_html, hearing_date=date(2026, 11, 8))
print(f"Parsed {len(entries)} hearing entries from the cause-list:\n")
for e in entries:
    print(f"  {e.case_no:16s} | {e.case_title_raw}")
    print(f"  {'':16s} | purpose: {e.purpose_raw}\n")

Parsed 4 hearing entries from the cause-list:

  CS/4521/2025     | M/s Sharma Textiles Pvt Ltd vs Rajendra Kumar
                   | purpose: For Hearing - Petitioner to file rejoinder

  CS/3390/2024     | Global Logistics Ltd vs Metro Warehousing Co.
                   | purpose: For Orders - Respondent to file reply within two weeks

  ARB/112/2026     | Northstar Fintech Pvt. Ltd. vs Anil Deshmukh & Anr.
                   | purpose: Final Arguments - Reserved for Judgment

  CS/9981/2023     | Bright Future Manufacturing vs State Electricity Board
                   | purpose: For Hearing - Petitioner's counsel absent, next date to be fixed



## 9. Run AI extraction + build the case store

Watch the 4th case (`CS/9981/2023`, "counsel absent, next date to be
fixed") -- the rule-based classifier correctly returns `next_action_on =
Unknown` rather than guessing wrong, because that phrasing doesn't match
any of its patterns. That's exactly the case a real pipeline would route
to an LLM call (set `ANTHROPIC_API_KEY` below to see that path run).

In [10]:
# Optional: paste an Anthropic API key to let low-confidence rows get
# reclassified by the LLM instead of staying "Unknown". Leave blank to see
# the pure rule-based behavior (the honest default, no cost, no key needed).
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # <-- uncomment to enable

store = CaseStore()
cases = store.ingest_hearing_entries(entries, api_key=os.environ.get("ANTHROPIC_API_KEY"))

for c in cases:
    print(f"{c.case_no:16s} status={c.status:22s} next_action_on={c.next_action_on:12s} court={c.court} | {c.judge}")

CS/4521/2025     status=Open                   next_action_on=Petitioner   court=Court No. 7 | Hon'ble Smt. Justice A. Verma
CS/3390/2024     status=Open                   next_action_on=Respondent   court=Court No. 3 | Hon'ble Shri Justice K. Mehta
ARB/112/2026     status=Reserved for Judgment  next_action_on=Court        court=Court No. 7 | Hon'ble Smt. Justice A. Verma
CS/9981/2023     status=Open                   next_action_on=Unknown      court=Court No. 1 | Hon'ble Shri Justice R. Nair


## 10. Add sample legal-spend entries and compute analytics

In [11]:
spend_entries = [
    SpendEntry("CS/4521/2025", "Filing", 15000, date(2025, 3, 1), "Court filing fee"),
    SpendEntry("CS/4521/2025", "Hearing", 8000, date(2025, 6, 1), "Counsel appearance fee"),
    SpendEntry("CS/3390/2024", "Filing", 22000, date(2024, 9, 1), "Court filing fee"),
    SpendEntry("CS/3390/2024", "Hearing", 12000, date(2025, 2, 1), "Counsel appearance fee"),
    SpendEntry("ARB/112/2026", "Hearing", 45000, date(2026, 5, 1), "Arbitrator + counsel fees"),
    SpendEntry("CS/9981/2023", "Filing", 9000, date(2023, 4, 1), "Court filing fee"),
    SpendEntry("CS/9981/2023", "Hearing", 18000, date(2024, 1, 1), "Counsel appearance fee"),
    SpendEntry("CS/9981/2023", "Hearing", 18000, date(2025, 1, 1), "Counsel appearance fee"),
]

summary = summarize_portfolio(cases, spend_entries, as_of=date(2026, 11, 8))

print("Status distribution  :", summary.status_counts)
print("Aging buckets         :", summary.aging_buckets)
print("Next-action breakdown :", summary.next_action_counts)
print(f"Total spend           : Rs {summary.total_spend_inr:,.0f}")
print("Spend by stage        :", summary.spend_by_stage)

Status distribution  : {'Open': 3, 'Reserved for Judgment': 1}
Aging buckets         : {'0-90 days': 4, '91-365 days': 0, '365+ days': 0}
Next-action breakdown : {'Petitioner': 1, 'Respondent': 1, 'Court': 1, 'Unknown': 1}
Total spend           : Rs 147,000
Spend by stage        : {'Filing': 46000.0, 'Hearing': 101000.0}


## 11. Generate the frontend HTML dashboard

This writes one self-contained `.html` file -- the dashboard a real
in-house legal team would actually look at. Open it in any browser.

In [12]:
import os
OUTPUT_DIR = "./legalops_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dashboard_path = os.path.join(OUTPUT_DIR, "legalops_dashboard.html")
with open(dashboard_path, "w") as f:
    f.write(render_dashboard_html(cases, summary))

print("Wrote:", dashboard_path)

Wrote: ./legalops_output\legalops_dashboard.html


### Preview inline (optional)

In [13]:
from IPython.display import IFrame, display
display(IFrame(src=dashboard_path, width="100%", height=750))

---
## What this notebook actually teaches

1. **Scraping real-world messy data** needs defensive parsing (null checks,
   regex normalization, graceful skipping of malformed rows) because the
   source data is never as clean as you'd like.
2. **"AI-powered" features in production are usually dual-mode**: fast,
   free, deterministic rules handle the common cases; an LLM is reserved
   for the cases rules can't confidently classify. Calling an LLM on every
   single record at scale is slow and expensive -- a well-engineered
   system avoids that by design.
3. **Separation of concerns** (scraper / AI module / case store / analytics
   / dashboard, each independently testable) is what lets a real
   engineering team scale a system like this without every change breaking
   everything else.
4. **Not everything labeled "AI" is machine learning.** The analytics layer
   here is 100% plain aggregation -- knowing the difference is itself a
   useful, demonstrable skill.

## Honest scope note
This is a teaching artifact built to understand the pattern behind
products like Provakil's ELM software -- it is not reverse-engineered from
their actual (private) source code, and it is not a production legal
product. Don't rely on its legal classifications for real case management
without review by a qualified professional.
